# TURRET Kalman Filter Analysis Notebook

## Overview

Comprehensive analysis notebook for TURRET captured aircraft tracking and Kalman filter prediction data.

### Sections

1. **Setup and Imports**
2. **Data Loading** - Load captured session data
3. **Data Exploration** - Basic statistics and data quality
4. **Aircraft Tracking Analysis** - Track trajectories and continuity
5. **Kalman Filter Performance** - Covariance, errors, state transitions
6. **Statistical Analysis** - Aircraft counts, rates, stability
7. **Temporal Analysis** - Update intervals, timing, anomalies
8. **Configuration Analysis** - Compare configs and generate recommendations

### Setup

Before running:
- Activate the virtual environment: `source /home/leon/DEV/DOCKER/TURRET/notebooks/venv/bin/activate`
- Ensure capture data exists in the `captures/` directory

In [ ]:
# Setup and Imports
import os
import sys
from pathlib import Path
from datetime import datetime

# Add modules to path
module_path = str(Path.cwd() / 'modules')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Import analysis modules
from modules.data_loader import DataLoader, create_data_loader
from modules.analysis import KalmanAnalysis, kalman_analysis
from modules.visualization import KalmanVisualizer, kalman_visualizer

# Core data science
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Setup complete!')
print(f'Module path: {module_path}')
print(f'Current directory: {Path.cwd()}')

In [ ]:
# Define captures directory
captures_dir = os.path.abspath(os.path.join('..', '..', 'captures'))

print(f'Captures directory: {captures_dir}')
print(f'Exists: {os.path.exists(captures_dir)}')

# Create data loader
data_loader = create_data_loader(captures_dir)
print('Data loader created!')

## 1. Data Loading

Load captured data from the TURRET CSV storage backend.

In [ ]:
# Find available dates
available_dates = data_loader.find_available_dates()
print('Available dates:')
for date in available_dates:
    print(f'  - {date}')

# Find sessions for latest date
if available_dates:
    latest_date = available_dates[0]
    sessions = data_loader.find_sessions_for_date(latest_date)
    print(f'
Sessions for {latest_date}:')
    for session in sessions:
        print(f'  - {os.path.basename(session)}')

In [ ]:
# Load all data
print('Loading data...')
predictions, opensky, statistics, metadata_list = data_loader.load_all_data()

print(f'Predictions: {len(predictions)} records')
print(f'OpenSky: {len(opensky)} records')
print(f'Statistics: {len(statistics)} records')
print(f'Sessions: {len(metadata_list)}')

# Show sample data
print('
=== Predictions Sample ===')
display(predictions.head(3))

print('
=== OpenSky Sample ===')
display(opensky.head(3))

In [ ]:
# Get combined summary
summary = data_loader.get_combined_summary(predictions, opensky, statistics, metadata_list)

print(f'Sessions: {summary["num_sessions"]}')
print(f'Total predictions: {summary["total_predictions"]}')
print(f'Unique callsigns: {summary["unique_callsigns_predictions"]}')
print(f'States: {summary["state_distribution"]}')

# Kalman configs
for i, config in enumerate(summary['kalman_configs']):
    print(f'
Config {i+1}:')
    for k, v in config.items():
        print(f'  {k}: {v}')

## 2. Data Exploration

Basic statistics and data quality analysis.

In [ ]:
# Basic statistics
pred_stats = kalman_analysis.get_basic_statistics(predictions)
os_stats = kalman_analysis.get_basic_statistics(opensky)

print('=== Predictions Statistics ===')
for col, stats in list(pred_stats.items())[:5]:
    print(f'{col}: mean={stats["mean"]:.2f}, std={stats["std"]:.2f}')

print('
=== OpenSky Statistics ===')
for col, stats in list(os_stats.items())[:5]:
    print(f'{col}: mean={stats["mean"]:.2f}, std={stats["std"]:.2f}')

In [ ]:
# Identify data gaps
pred_gaps = kalman_analysis.identify_data_gaps(predictions)
os_gaps = kalman_analysis.identify_data_gaps(opensky)

print('=== Predictions Gaps ===')
print(f'Total records: {pred_gaps["total_records"]}')
print(f'Missing values: {pred_gaps["total_missing_values"]}')
print(f'Avg time between records: {pred_gaps["avg_time_between_records"]:.4f}s')
print(f'Max gap: {pred_gaps["max_gap"]:.4f}s')
print(f'Gaps > 1s: {pred_gaps["gaps_greater_than_1s"]}')
print(f'Gaps > 5s: {pred_gaps["gaps_greater_than_5s"]}')
print(f'Gaps > 10s: {pred_gaps["gaps_greater_than_10s"]}')

In [ ]:
# Plot distributions
key_cols = ['latitude', 'longitude', 'baro_altitude', 'velocity']
plot_cols = [c for c in key_cols if c in predictions.columns]

fig, axes = kalman_visualizer.plot_distribution_histograms(predictions, plot_cols, bins=30)
plt.suptitle('Predictions Data Distributions')
plt.tight_layout()
plt.show()

## 3. Aircraft Tracking Analysis

Track individual aircraft and analyze trajectories.

In [ ]:
# Get trajectories
trajectories = kalman_analysis.get_aircraft_trajectories(predictions)
print(f'Found {len(trajectories)} aircraft trajectories')

# Show first few
for callsign in list(trajectories.keys())[:5]:
    print(f'{callsign}: {len(trajectories[callsign])} predictions')

In [ ]:
# Plot trajectories
if trajectories:
    sample_callsigns = list(trajectories.keys())[:min(10, len(trajectories))]
    sample_preds = predictions[predictions['callsign'].isin(sample_callsigns)]
    
    fig, ax = kalman_visualizer.plot_aircraft_trajectories(
        sample_preds, callsigns=sample_callsigns, figsize=(14, 10)
    )
    plt.show()

In [ ]:
# Track continuity analysis
continuity_results = []
for callsign, trajectory in trajectories.items():
    metrics = kalman_analysis.calculate_track_continuity(trajectory)
    continuity_results.append(metrics)

# Summary
if continuity_results:
    coverages = [m.track_coverage for m in continuity_results]
    print(f'Average track coverage: {np.mean(coverages):.2%}')
    print(f'Median track coverage: {np.median(coverages):.2%}')
    print(f'Average frames per aircraft: {np.mean([m.total_frames for m in continuity_results]):.1f}')

In [ ]:
# Track accuracy analysis
accuracy_metrics = kalman_analysis.analyze_track_accuracy(predictions, opensky)
print(f'Aircraft with comparison data: {len(accuracy_metrics)}')

# Overall error statistics
error_stats = kalman_analysis.calculate_prediction_error_statistics(predictions, opensky)
if error_stats:
    print(f'\nRMSE Position: {error_stats["rmse_position_m"]:.2f}m')
    print(f'MAE Position: {error_stats["mae_position_m"]:.2f}m')
    print(f'RMSE Altitude: {error_stats["rmse_altitude_m"]:.2f}m')
    print(f'RMSE Velocity: {error_stats["rmse_velocity_knots"]:.2f} knots')

In [ ]:
# 3D trajectory plot
if trajectories:
    sample_callsign = list(trajectories.keys())[0]
    trajectory = trajectories[sample_callsign]
    
    fig = kalman_visualizer.plot_3d_trajectory(trajectory, callsign=sample_callsign)
    plt.show()

## 4. Kalman Filter Performance

Analyze covariance, prediction errors, and state transitions.

In [ ]:
# Covariance convergence analysis
covariance_results = []
for callsign, trajectory in trajectories.items():
    if len(trajectory) >= 5:
        result = kalman_analysis.analyze_covariance_convergence(trajectory)
        covariance_results.append(result)

print(f'Analyzed {len(covariance_results)} aircraft')

# Summary
if covariance_results:
    rates = [r.convergence_rate for r in covariance_results]
    steady = [r.steady_state_reached for r in covariance_results]
    print(f'Avg convergence rate: {np.mean(rates):.2%}')
    print(f'Steady state reached: {sum(steady)}/{len(steady)} ({sum(steady)/len(steady)*100:.1f}%)')

In [ ]:
# Plot covariance
if covariance_results:
    best_idx = np.argmax([r.convergence_rate for r in covariance_results])
    best_callsign = covariance_results[best_idx].callsign
    best_traj = trajectories[best_callsign]
    
    fig, ax = kalman_visualizer.plot_covariance_over_time(best_traj, callsign=best_callsign)
    plt.show()

In [ ]:
# State transition analysis
state_data = kalman_analysis.analyze_state_transitions(predictions)

print('Transition counts:')
for transition, count in state_data['transition_counts'].items():
    print(f'  {transition}: {count}')

print('State durations (s):')
for state, duration in state_data['state_durations'].items():
    print(f'  {state}: {duration:.2f}')

# Plot transition matrix
if not state_data['state_transition_matrix'].empty:
    fig, ax = kalman_visualizer.plot_state_transition_matrix(state_data['state_transition_matrix'])
    plt.show()

In [ ]:
# Velocity vector accuracy
vel_metrics = kalman_analysis.analyze_velocity_vector_accuracy(predictions, opensky)

if vel_metrics:
    print(f'V North RMSE: {vel_metrics["v_north_rmse"]:.4f}')
    print(f'V East RMSE: {vel_metrics["v_east_rmse"]:.4f}')
    print(f'Velocity vector RMSE: {vel_metrics["velocity_vector_rmse"]:.4f}')

## 5. Statistical Analysis

Aircraft counts, prediction rates, and stability metrics.

In [ ]:
# Aircraft count over time
count_data = kalman_analysis.analyze_aircraft_count(statistics)

if len(count_data) > 0:
    fig, ax = kalman_visualizer.plot_aircraft_count(count_data)
    plt.show()
    
    print(f'Max count: {count_data["count"].max():.0f}')
    print(f'Mean count: {count_data["count"].mean():.1f}')
else:
    print('No aircraft count data')

In [ ]:
# Prediction measurement rates
rate_data = kalman_analysis.analyze_prediction_measurement_rates(statistics)

if rate_data:
    print(f'Avg time between frames: {rate_data["avg_time_between_frames"]:.4f}s')
    print(f'Avg predictions per frame: {rate_data["avg_predictions_per_frame"]:.2f}')
    print(f'Total predictions: {rate_data["total_predictions"]}')
    print(f'Total measurements: {rate_data["total_measurements"]}')

In [ ]:
# Track stability metrics
stability = kalman_analysis.calculate_track_stability_metrics(predictions)

if stability:
    print(f'Avg position jump: {stability["avg_position_jump_m"]:.2f}m')
    print(f'Max position jump: {stability["max_position_jump_m"]:.2f}m')
    print(f'Large jumps (>1km): {stability["large_jumps_count"]}')

In [ ]:
# Altitude distribution
alt_data = kalman_analysis.analyze_altitude_distribution(predictions)

if alt_data:
    print(f'Mean altitude: {alt_data["mean_altitude_m"]:.2f}m')
    print(f'Max altitude: {alt_data["max_altitude_m"]:.2f}m')
    print(f'Max climb rate: {alt_data["max_climb_rate_mps"]:.2f}m/s')
    print(f'Max descent rate: {alt_data["max_descent_rate_mps"]:.2f}m/s')

In [ ]:
# Spatial density heatmap
if len(predictions) > 0:
    fig, ax = kalman_visualizer.plot_spatial_density_heatmap(predictions, bins=50)
    plt.show()

## 6. Temporal Analysis

Update intervals, timing consistency, and anomalies.

In [ ]:
# Update interval analysis
if len(statistics) > 0:
    interval_data = kalman_analysis.analyze_update_intervals(statistics)
else:
    mock_stats = predictions.groupby('timestamp').size().reset_index(name='track_count')
    mock_stats['prediction_count'] = mock_stats['track_count']
    mock_stats['total_predictions'] = mock_stats['track_count'].cumsum()
    mock_stats['total_measurements'] = mock_stats['track_count'].cumsum()
    mock_stats['frame_index'] = range(len(mock_stats))
    mock_stats['session_id'] = predictions['session_id'].iloc[0]
    interval_data = kalman_analysis.analyze_update_intervals(mock_stats)

if interval_data:
    print(f'Mean interval: {interval_data["mean_interval_s"]:.4f}s')
    print(f'Std interval: {interval_data["std_interval_s"]:.4f}s')
    print(f'Intervals > 1s: {interval_data["intervals_greater_than_1s"]}')
    print(f'Intervals > 5s: {interval_data["intervals_greater_than_5s"]}')

In [ ]:
# Timing consistency
if len(statistics) > 0:
    timing = kalman_analysis.analyze_frame_timing_consistency(statistics)
else:
    timing = kalman_analysis.analyze_frame_timing_consistency(mock_stats)

if timing:
    print(f'Expected interval: {timing["expected_interval_s"]:.4f}s')
    print(f'Jitter coefficient: {timing["jitter_coefficient"]:.2%}')

In [ ]:
# Timing anomalies
if len(statistics) > 0:
    anomalies = kalman_analysis.identify_timing_anomalies(statistics)
else:
    anomalies = kalman_analysis.identify_timing_anomalies(mock_stats)

print(f'Anomalous intervals: {len(anomalies)}')

if len(anomalies) > 0:
    fig, ax = kalman_visualizer.plot_timing_anomalies(anomalies)
    plt.show()

## 7. Configuration Analysis

Compare configurations and generate tuning recommendations.

In [ ]:
# Compare configurations
config_comp = kalman_analysis.compare_configurations(metadata_list)

print(f'Unique configurations: {config_comp["unique_configs"]}')

for i, config in enumerate(config_comp['configurations']):
    print(f'\nConfig {i+1}: {config["session_count"]} sessions')
    print(f'  Process Noise: {config["position_process_noise"]}')
    print(f'  Measurement Noise: {config["position_measurement_noise"]}')

In [ ]:
# Generate recommendations
recommendations = kalman_analysis.generate_configuration_recommendations(
    predictions, opensky, metadata_list
)

print('=== Configuration Recommendations ===')
if recommendations:
    for param, rec in recommendations['recommendations'].items():
        priority = recommendations['priority'].get(param, 'LOW')
        print(f'\n[{priority}] {param}: {rec}')

In [ ]:
# Process noise impact
noise_data = kalman_analysis.analyze_process_noise_impact(predictions, metadata_list)

if noise_data:
    print('Process noise levels:', noise_data['noise_levels'])
    for noise, metrics in noise_data['metrics'].items():
        print(f'Noise={noise}: Avg Lat Cov={metrics["avg_covariance_lat"]:.2e}')

## 8. Next Steps

### What to do next:

1. Review analysis results and recommendations
2. Tune Kalman filter parameters based on recommendations
3. Run new capture sessions with updated configurations
4. Re-run this notebook to compare performance

### Resources:

- [Kalman Filter Theory](https://en.wikipedia.org/wiki/Kalman_filter)
- TURRET Documentation

---

**Notebook Version**: 1.0.0 | **Last Updated**: 2026-08-24